In [68]:
# %pip install mlflow

In [69]:
import mlflow
mlflow.set_experiment("Spotify Quickstart")

import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

In [70]:
mlflow.sklearn.autolog()

In [71]:
df = pd.read_csv("data/dataset_copy.csv").drop(["track_num"],axis=1).dropna(subset = ['track_name'])
df.head()

,track_id,artist_name,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson & ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [72]:
df["artist_track_name"] = df["artist_name"] + " - "+ df["track_name"]
df.drop_duplicates(subset=['artist_track_name'], keep=False)


,track_id,artist_name,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,genre,artist_track_name
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson & ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.359,0,...,1,0.0557,0.210,0.000000,0.1170,0.1200,76.332,4,acoustic,Ingrid Michaelson & ZAYN - To Begin Again
14,4LbWtBkN82ZRhz9jqzgrb3,Chord Overstreet & Deepend,Hold On (Remix),Hold On - Remix,56,188133,False,0.755,0.780,2,...,1,0.0327,0.124,0.000028,0.1210,0.3870,120.004,4,acoustic,Chord Overstreet & Deepend - Hold On - Remix
19,6CgNoAbFJ4Q4Id4EjtbXlC,Boyce Avenue & Bea Miller,"Cover Sessions, Vol. 4",Photograph,67,260186,False,0.717,0.320,3,...,1,0.0283,0.830,0.000000,0.1070,0.3220,107.946,4,acoustic,Boyce Avenue & Bea Miller - Photograph
21,210JCw2LbYD4YIs8GiZ9iP,Boyce Avenue & Jennel Garcia,"Cover Sessions, Vol. 3",Demons,63,174174,False,0.678,0.351,0,...,1,0.0266,0.747,0.000000,0.3550,0.5690,90.032,4,acoustic,Boyce Avenue & Jennel Garcia - Demons
32,1m5LC29RE52Bxy7hxvpOlL,Chord Overstreet,Christmas Country Songs 2022,All I Want For Christmas Is A Real Good Tan,0,234186,False,0.593,0.455,6,...,1,0.0388,0.366,0.000000,0.0914,0.5640,202.019,4,acoustic,Chord Overstreet - All I Want For Christmas Is...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113995,2C3TZjDRiAzdyViavDJ217,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Sleep My Little Boy,21,384999,False,0.172,0.235,5,...,1,0.0422,0.640,0.928000,0.0863,0.0339,125.995,5,world-music,Rainy Lullaby - Sleep My Little Boy
113996,1hIz5L4IB9hN3WRYPOCGPw,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Water Into Light,22,385000,False,0.174,0.117,0,...,0,0.0401,0.994,0.976000,0.1050,0.0350,85.239,4,world-music,Rainy Lullaby - Water Into Light
113997,6x8ZfSoqDjuNa5SVP5QjvX,Cesária Evora,Best Of,Miss Perfumado,22,271466,False,0.629,0.329,0,...,0,0.0420,0.867,0.000000,0.0839,0.7430,132.378,4,world-music,Cesária Evora - Miss Perfumado
113998,2e6sXL2bYv4bSz6VTdnfLs,Michael W. Smith,Change Your World,Friends,41,283893,False,0.587,0.506,7,...,1,0.0297,0.381,0.000000,0.2700,0.4130,135.960,4,world-music,Michael W. Smith - Friends


In [73]:
# since genre is in the form of different unique genres
mylist=df['artist_track_name'].to_list()
mylist = list(dict.fromkeys(mylist))

In [74]:
genre_encoded = pd.get_dummies(df["genre"])

In [75]:
features = df[[
    'danceability', 'energy', 'loudness', 'instrumentalness', 'tempo', 
    'key', 'mode', 'popularity'
    # ,'duration_ms'
]]

X = pd.concat([features, genre_encoded], axis=1)

In [76]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [85]:
knn = NearestNeighbors(
    n_neighbors=10,
    metric="euclidean"
)

knn.fit(X_scaled)


model_info =mlflow.sklearn.log_model(knn,"KNN model")

2026/05/26 12:57:55 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'd62800faea744109b952a381d85b1df6', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


2026/05/26 12:57:55 WARNING mlflow.sklearn: Failed to infer model signature: the trained model does not have a `predict` or `transform` function, which is required in order to infer the signature
2026/05/26 12:57:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/26 12:57:55 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2026/05/26 12:57:58 WARNING mlflow.sklearn: Training metrics will not be recorded because training labels were not specified. To automatically record training metrics, provide training labels as inputs to the model training function.
2026/05/26 12:57:58 WARNING mlflow.models.model: `ar

In [78]:
# lookup track by track_name
def get_song_index(song_name):
    return df[df["artist_track_name"] == song_name].index[0]

In [ ]:
# get la neighbors for that song
def get_neighbors(song_name):
    index = get_song_index(song_name)
    song_vector = X_scaled[index].reshape(1,-1)
    distances, indices = knn.kneighbors(song_vector)

    return distances[0], indices[0]

In [80]:
def genre_bonus(index, song_A, song_B):
    genre_A = df[df["artist_track_name"] == song_A]["genre"].values[0]
    genre_B = df[df["artist_track_name"] == song_B]["genre"].values[0]

    genre_C = df.iloc[index]["genre"]

    bonus = 0

    if genre_C == genre_A:
        bonus += 0.1
    if genre_C == genre_B:
        bonus += 0.1

    return bonus

In [81]:
def bridge_recommendation(song_A, song_B, top_n=5): 
    dist_A, ind_A = get_neighbors(song_A) # the distance and the independent variables chosen
    dist_B, ind_B = get_neighbors(song_B)

    scores = {}

    # convert those distances into similarity
    for d, i in zip(dist_A, ind_A):
        # scores[i] = scores.get(i, 0) + (1 / (1 + d)) # without the added weight of genre(OneHotEncoding)
        scores[i] = scores.get(i, 0) + (1 / (1 + d)) + genre_bonus(i, song_A, song_B)

    for d, i in zip(dist_B, ind_B):
        # scores[i] = scores.get(i, 0) + (1 / (1 + d))
        scores[i] = scores.get(i, 0) + (1 / (1 + d)) + genre_bonus(i, song_A, song_B)

    # sort by combined score
    sorted_songs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    
    results = []
    for i, score in sorted_songs:
        name= df.iloc[i]["artist_track_name"]

        if name not in [song_A, song_B]:
            results.append((name, score))

    return results[:top_n]

In [82]:
def normalize_scores(results):
    if not results:
        return []
    
    max_score = max(score for _, score in results)

    output = []
    for name, score in results:
        confidence = score / max_score
        output.append((name, score, confidence))

    return output

In [83]:
def getresults(songA, songB):
    results = bridge_recommendation(songA, songB,4)
    normalized_results = normalize_scores(results)
    for name, score, confidence in normalized_results:
        print(f"{name} → Score: {score:.4f} | Confidence: {confidence:.2%}")

In [84]:
getresults("Alan Walker - Faded - Lost Stories Remix","Caravan Palace - Lone Digger")

Snakehips & ZAYN - Cruel (feat. ZAYN) → Score: 0.6154 | Confidence: 100.00%
Shankar & Ehsaan & Loy & Sonu Nigam & Mahalakshmi Iyer & Gulzar - Chup Chup Ke → Score: 0.5616 | Confidence: 91.26%
DJ Snake & Marshmello & Justin Bieber - Let Me Love You - Marshmello Remix → Score: 0.5515 | Confidence: 89.62%
Dimitri Vegas & Like Mike & David Guetta & Dimitri Vegas & Kiiara - Complicated (feat. Kiiara) → Score: 0.5423 | Confidence: 88.12%


In [87]:
loaded_model =mlflow.sklearn.load_model(
    model_info.model_uri
)

In [88]:
loaded_model

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'euclidean'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [ ]:
#postman